In [1]:
## 10-40 Data Chromatogram Plotting - Make HTMLs - Calculate RT's - Hyperlink to spreadsheet

import pandas as pd
import numpy as np
import os
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from tqdm.auto import tqdm
import plotly.graph_objects as go
from openpyxl import load_workbook
from openpyxl.styles import Font

def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-((x - mean) / (2 * stddev))**2)

def add_hyperlinks(excel_file, plots_folder):
    wb = load_workbook(excel_file)
    ws = wb.active
    
    # Find or create the "Plot Link" column
    plot_link_col = None
    for col in range(1, ws.max_column + 1):
        if ws.cell(row=1, column=col).value == "Plot Link":
            plot_link_col = col
            break
    
    if plot_link_col is None:
        plot_link_col = ws.max_column + 1
        ws.cell(row=1, column=plot_link_col, value="Plot Link")
    
    for row in ws.iter_rows(min_row=2, max_col=1):
        common_name = row[0].value
        if common_name:
            html_file = f"{common_name}.html"
            file_path = os.path.join(plots_folder, html_file)
            if os.path.exists(file_path):
                cell = ws.cell(row=row[0].row, column=plot_link_col)
                cell.value = "View Plot"
                cell.hyperlink = file_path
                cell.font = Font(color="0000FF", underline="single")
    
    wb.save(excel_file)

# Define input spreadsheet, output folder, and output .xlsx file
excelsheet = "/Users/grant/Library/CloudStorage/GoogleDrive-brodohfasho@gmail.com/Shared drives/DEL Lipophilicity Paper/Data Spreadsheets/10-40_Testset.xlsx"
workbook = "Sheet1"
output_folder = "/Users/grant/Desktop/Test/"
plots_folder = os.path.join(output_folder, "plots")
os.makedirs(output_folder, exist_ok=True)
os.makedirs(plots_folder, exist_ok=True)
newdata = os.path.join(output_folder, os.path.basename(excelsheet).replace('.xlsx', '_RTs.xlsx'))

# Define peak picking parameters
stddev_threshold = 2
min_height_threshold_factor = 0.35
fit_width = 1.5
minimum_RT = 10

# Create pandas dataframe from sequencing data and define indexes
df = pd.read_excel(excelsheet, sheet_name=workbook)
df.set_index(["Common_Name (N-->C)", "lid"], inplace=True)

# Group by the first level of the MultiIndex
grouped_df = df.groupby(level=0)

# Create a dictionary for RT data collection
out_dict = {}

# Iterate over grouped data
for index, group_df in tqdm(grouped_df):
    out_dict[index] = list()
    fig = go.Figure()

    for (lid, row) in group_df.iterrows():
        # Check if the "max_count" value is less than 100
        if row["max_count"] < 20:
            continue  # Skip the Gaussian fitting and plotting for this entry

        # Define new dataframe of delimited RT_count data
        chrom_df = pd.Series(row["all_datapoints"])

        # Split the comma-delimited values into a new dataframe (time, counts)
        chromvalues_df = chrom_df.str.split(",", expand=True)
        chromvalues_new = chromvalues_df.transpose()
        master_df = chromvalues_new[0].str.split("[:,;]", expand=True)

        # Create a DataFrame with the time and count values
        data = pd.DataFrame({'Time': master_df[0].astype(float) / 60, 'Count': master_df[2].astype(float)})

        # Sort the DataFrame based on the 'Time' column
        data = data.sort_values('Time')

        # Extract the sorted time and count values
        x = data['Time'].tolist()
        y = data['Count'].tolist()

        # Find peaks with a minimum height
        peaks, _ = find_peaks(y, height=max(y) * min_height_threshold_factor)

        gaussian_means = []

        # Fit and plot Gaussian for each peak
        for peak in peaks:
            # Define a range around the peak to fit the Gaussian
            # Determine the indices for fitting
            indices = [i for i, val in enumerate(x) if x[peak] - fit_width < val < x[peak] + fit_width]
            if not indices:
                continue

            fit_x = [x[i] for i in indices]
            fit_y = [y[i] for i in indices]

            # Ensure there are enough points to fit
            if len(fit_x) < 3:
                continue
            if x[peak] < minimum_RT:
                continue

            # Initial guess: amplitude, mean, stddev
            initial_guess = [max(fit_y), x[peak], np.std(fit_x)/2]
            bounds = ([0, x[peak] - fit_width, 0], [np.inf, x[peak] + fit_width, np.inf])

            try:
                popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)
                if popt[2] < stddev_threshold:
                    color = 'navy' if 'DEL-0044' in lid else 'firebrick'
                    label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'
                    fit_x_values = np.linspace(min(fit_x)-0.5, max(fit_x)+0.5, 100)
                    w = gaussian(fit_x_values, *popt)
                    gaussian_means.append((popt[1], popt[0]))
                    fig.add_trace(go.Scatter(x=fit_x_values, y=w, mode='lines', line=dict(color=color, dash='dash', width=2), name=f"{label} Gaussian Fit", visible='legendonly'))
            except RuntimeError as e:
                pass

        best_mean = minimum_RT
        best_amplitude = 0
        for mean, amplitude in gaussian_means:
            if mean > best_mean:
                best_mean = mean
                best_amplitude = amplitude

        fig.add_annotation(x=best_mean, y=best_amplitude, text=f'Centroid RT ({lid[1]}): {best_mean:.2f}', showarrow=True, arrowhead=1, axref='x', ayref='y', ax=best_mean+8, ay=best_amplitude, font=dict(size=12))

        if 'DEL-0044' not in lid:
            out_dict[index].append(best_mean)
        else:
            out_dict[index].insert(0, best_mean)

        # Customize the appearance of each line
        color = 'plum' if 'DEL-0044' in lid else 'coral'
        label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'

        # Plot the data as a scatter plot with a connecting line
        fig.add_trace(go.Scatter(x=x, y=y, mode='markers+lines', marker=dict(color=color, size=10), line=dict(color=color, width=3), name=label, visible=True))

    # Customize legend and other plot properties
    fig.update_layout(
        legend=dict(
            x=1,
            y=1,
            font=dict(size=16),
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='rgba(0, 0, 0, 0.2)',
            borderwidth=1,
            itemclick='toggleothers',
            itemdoubleclick='toggle'
        ),
        xaxis=dict(title='Time (min)', titlefont=dict(size=16), range=[0, 65], dtick=5),
        yaxis=dict(title='Scaled_counts', titlefont=dict(size=16)),
        title=dict(text=index, font=dict(size=18)),
        margin=dict(l=20, r=20, t=60, b=20),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    plot_filename = os.path.join(plots_folder, f'{index}.html')
    fig.write_html(plot_filename)

outDF = pd.DataFrame(out_dict).T
outDF.rename(columns={0: 'Linear RT (min)', 1: 'Cyclized RT (min)'}, inplace=True)
outDF.index.name = 'Common Name'

out_dict = {k: [x, y, y-x] for k, (x, y) in out_dict.items()}
Linear_RTs = list()
Cyclized_RTs = list()
deltaRT = list()
for ix0, ix1 in df.index:
    try:
        if 'DEL-0044' in ix1:
            Linear_RTs.append(out_dict[ix0][0])
            Cyclized_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
        elif 'DEL-0045' in ix1:
            Cyclized_RTs.append(out_dict[ix0][1])
            Linear_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
    except KeyError:
        print(f"DIDN'T FIND {ix0},{ix1}. setting to 'np.nan'")
        Linear_RTs.append(np.nan)
        Cyclized_RTs.append(np.nan)
        deltaRT.append(np.nan)
df['Linear RT (min)'] = Linear_RTs
df['Cyclized RT (min)'] = Cyclized_RTs
df['Delta RT (min) (Cyclized-Linear)'] = deltaRT

df.to_excel(newdata, merge_cells=False, index=True)

# Add hyperlinks to the Excel file
add_hyperlinks(newdata, plots_folder)

print(f"Excel file with hyperlinks saved as: {newdata}")
print(f"Plots saved in: {plots_folder}")

  0%|          | 0/2 [00:00<?, ?it/s]

ValueError: All arrays must be of the same length

In [2]:
## 10-60 Processing with Plot.ly
## Adds hyperlinks - in beta - delete comment if working

# Import packages as easy abbreviations
import pandas as pd
import numpy as np
import os
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from tqdm.auto import tqdm
import plotly.graph_objects as go
from openpyxl import load_workbook
from openpyxl.styles import Font

# Define input spreadsheet. Define output folder for image files and name for output .xlsx file
excelsheet = "/Users/grant/Library/CloudStorage/GoogleDrive-brodohfasho@gmail.com/Shared drives/DEL Lipophilicity Paper/Data Spreadsheets/10-60_Testset.xlsx"
workbook = "Sheet1"
output_folder = "/Users/grant/Desktop/Test2/"
plots_folder = os.path.join(output_folder, "plots")
os.makedirs(output_folder, exist_ok=True)
os.makedirs(plots_folder, exist_ok=True)
newdata = os.path.join(output_folder, os.path.basename(excelsheet).replace('.xlsx', '_RTs.xlsx'))

# Define peak picking parameters
stddev_threshold = 2
min_height_threshold_factor = 0.35
fit_width = 1.5
minimum_RT = 10

# Define gaussian equation
def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-((x - mean) / (2 * stddev))**2)

# Define function to add hyperlinks
def add_hyperlinks(excel_file, plots_folder):
    wb = load_workbook(excel_file)
    ws = wb.active
    
    # Find or create the "Plot Link" column
    plot_link_col = None
    for col in range(1, ws.max_column + 1):
        if ws.cell(row=1, column=col).value == "Plot Link":
            plot_link_col = col
            break
    
    if plot_link_col is None:
        plot_link_col = ws.max_column + 1
        ws.cell(row=1, column=plot_link_col, value="Plot Link")
    
    for row in ws.iter_rows(min_row=2, max_col=1):
        common_name = row[0].value
        if common_name:
            html_file = f"{common_name}.html"
            file_path = os.path.join(plots_folder, html_file)
            if os.path.exists(file_path):
                cell = ws.cell(row=row[0].row, column=plot_link_col)
                cell.value = "View Plot"
                cell.hyperlink = file_path
                cell.font = Font(color="0000FF", underline="single")
    
    wb.save(excel_file)

# Create pandas dataframe from sequencing data
# Define indexes
df = pd.read_excel(excelsheet, sheet_name=workbook)
df.set_index(["Common_Name (N-->C)", "lid"], inplace=True)

# Group by the first level of the MultiIndex
grouped_df = df.groupby(level=0)

# Create a dictionary for RT data collection
out_dict = {}

# Iterate over grouped data
for index, group_df in tqdm(grouped_df):
    out_dict[index] = list()
    fig = go.Figure()

    for (lid, row) in group_df.iterrows():
        # Define new dataframe of delimited RT_count data
        chrom_df = pd.Series(row["all_datapoints"])

        # Split the comma-delimited values into a new dataframe (time, counts)
        chromvalues_df = chrom_df.str.split(",", expand=True)
        chromvalues_new = chromvalues_df.transpose()
        master_df = chromvalues_new[0].str.split("[:,;]", expand=True)

        # Create a DataFrame with the time and count values
        data = pd.DataFrame({'Time': master_df[0].astype(float) / 60, 'Count': master_df[2].astype(float)})

        # Sort the DataFrame based on the 'Time' column
        data = data.sort_values('Time')

        # Extract the sorted time and count values
        x = data['Time'].tolist()
        y = data['Count'].tolist()

        # Find peaks with a minimum height
        peaks, _ = find_peaks(y, height=max(y) * min_height_threshold_factor)

        gaussian_means = []

        # Fit and plot Gaussian for each peak
        for peak in peaks:
            # Define a range around the peak to fit the Gaussian
            # Determine the indices for fitting
            indices = [i for i, val in enumerate(x) if x[peak] - fit_width < val < x[peak] + fit_width]
            if not indices:
                continue

            fit_x = [x[i] for i in indices]
            fit_y = [y[i] for i in indices]

            # Ensure there are enough points to fit
            if len(fit_x) < 3:
                continue
            if x[peak] < minimum_RT:
                continue

            # Initial guess: amplitude, mean, stddev
            initial_guess = [max(fit_y), x[peak], np.std(fit_x)/2]
            bounds = ([0, x[peak] - fit_width, 0], [np.inf, x[peak] + fit_width, np.inf])

            try:
                popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)
                if popt[2] < stddev_threshold:
                    color = 'navy' if 'DEL-0044' in lid else 'firebrick'
                    label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'
                    fit_x_values = np.linspace(min(fit_x)-0.5, max(fit_x)+0.5, 100)
                    w = gaussian(fit_x_values, *popt)
                    gaussian_means.append((popt[1], popt[0]))
                    fig.add_trace(go.Scatter(x=fit_x_values, y=w, mode='lines', line=dict(color=color, dash='dash', width=2), name=f"{label} Gaussian Fit", visible='legendonly'))
            except RuntimeError as e:
                pass

        best_mean = minimum_RT
        best_amplitude = 0
        for mean, amplitude in gaussian_means:
            if mean > best_mean:
                best_mean = mean
                best_amplitude = amplitude

        fig.add_annotation(x=best_mean, y=best_amplitude, text=f'Centroid RT ({lid[1]}): {best_mean:.2f}', showarrow=True, arrowhead=1, axref='x', ayref='y', ax=best_mean+8, ay=best_amplitude, font=dict(size=12))

        if 'DEL-0044' not in lid:
            out_dict[index].append(best_mean)
        else:
            out_dict[index].insert(0, best_mean)

        # Customize the appearance of each line
        color = 'plum' if 'DEL-0044' in lid else 'coral'
        label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'

        # Plot the data as a scatter plot with a connecting line
        fig.add_trace(go.Scatter(x=x, y=y, mode='markers+lines', marker=dict(color=color, size=10), line=dict(color=color, width=3), name=label, visible=True))

    # Customize legend and other plot properties
    fig.update_layout(
        legend=dict(
            x=1,
            y=1,
            font=dict(size=16),
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='rgba(0, 0, 0, 0.2)',
            borderwidth=1,
            itemclick='toggleothers',
            itemdoubleclick='toggle'
        ),
        xaxis=dict(title='Time (min)', titlefont=dict(size=16), range=[0, 65], dtick=5),
        yaxis=dict(title='Scaled_counts', titlefont=dict(size=16)),
        title=dict(text=index, font=dict(size=18)),
        margin=dict(l=20, r=20, t=60, b=20),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    plot_filename = os.path.join(plots_folder, f'{index}.html')
    fig.write_html(plot_filename)

outDF = pd.DataFrame(out_dict).T
outDF.rename(columns={0: 'Linear RT (min)', 1: 'Cyclized RT (min)'}, inplace=True)
outDF.index.name = 'Common Name'

out_dict = {k: [x, y, y-x] for k, (x, y) in out_dict.items()}
Linear_RTs = list()
Cyclized_RTs = list()
deltaRT = list()
for ix0, ix1 in df.index:
    try:
        if 'DEL-0044' in ix1:
            Linear_RTs.append(out_dict[ix0][0])
            Cyclized_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
        elif 'DEL-0045' in ix1:
            Cyclized_RTs.append(out_dict[ix0][1])
            Linear_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
    except KeyError:
        print(f"DIDN'T FIND {ix0},{ix1}. setting to 'np.nan'")
        Linear_RTs.append(np.nan)
        Cyclized_RTs.append(np.nan)
        deltaRT.append(np.nan)
df['Linear RT (min)'] = Linear_RTs
df['Cyclized RT (min)'] = Cyclized_RTs
df['Delta RT (min) (Cyclized-Linear)'] = deltaRT

df.to_excel(newdata, merge_cells=False, index=True)

# Add hyperlinks to the Excel file
add_hyperlinks(newdata, plots_folder)

print(f"Excel file with hyperlinks saved as: {newdata}")
print(f"Plots saved in: {plots_folder}")

  0%|          | 0/2 [00:00<?, ?it/s]

Excel file with hyperlinks saved as: /Users/grant/Desktop/Test2/10-60_Testset_RTs.xlsx
Plots saved in: /Users/grant/Desktop/Test2/plots


/var/folders/5m/wpxkkdws6gn53blbx7s7r2x40000gn/T/ipykernel_4758/535839975.py:125: OptimizeWarning:

Covariance of the parameters could not be estimated

/var/folders/5m/wpxkkdws6gn53blbx7s7r2x40000gn/T/ipykernel_4758/535839975.py:125: OptimizeWarning:

Covariance of the parameters could not be estimated

